# Fine Print — QLoRA fine-tune on Kaggle (v0.4, D37)

`MODE = 'timing'`: 30 optimizer steps with each backend — transformers+peft on **both** T4s (DDP) vs Unsloth on
one T4 — to measure speed before committing GPU hours. `MODE = 'full'`: train with the chosen backend, then
evaluate the adapter on `test-500` (GPU 0) and `lexglue` (GPU 1) at the same time.

Inputs: the private `fine-print` dataset (code + processed data, `fineprint.kaggle_bundle`). Needs GPU T4 x2 +
internet. Output: `fine-print-results.zip` (scores, predictions, training summary, logs) and, in full mode,
`fine-print-adapter.zip` (LoRA weights). Import on the Mac with `uv run python -m fineprint.kaggle_results`.

In [ ]:
MODE = 'full'       # 'timing' or 'full'
BACKEND = 'hf2'     # full mode: 'hf2' (transformers+peft, 2 GPUs) or 'unsloth' (1 GPU), from the timing run
EXTRA = ['--epochs', '1']  # D37: 1 epoch ≈ 4.6 h on 2×T4; full mode: extra train.py args, e.g. ['--epochs', '1'] or ['--limit', '20000']

import glob, os, shutil, subprocess, sys, time, zipfile
src = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/fine-print/configs', recursive=True))
REPO, W = '/kaggle/working/fine-print', '/kaggle/working'
if not os.path.exists(REPO):
    shutil.copytree(src, REPO)
os.chdir(REPO)
ENV = {**os.environ, 'PYTHONPATH': f'{REPO}/src', 'TOKENIZERS_PARALLELISM': 'false',
       'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}  # less memory fragmentation
subprocess.run(['nvidia-smi', '-L'])

In [ ]:
def sh(cmd, log, gpus='0,1', check=False):
    """Run a shell command with its output in a log file; print the last line every 2 minutes."""
    with open(f'{W}/{log}', 'a') as f:
        p = subprocess.Popen(['bash', '-c', cmd], stdout=f, stderr=subprocess.STDOUT,
                             env={**ENV, 'CUDA_VISIBLE_DEVICES': gpus})
    while p.poll() is None:
        time.sleep(120)
        lines = [l for l in open(f'{W}/{log}', errors='replace').read().splitlines() if l.strip()]
        print(f'[{log}]', lines[-1][:160] if lines else '(starting)', flush=True)
    print(f'[{log}] exit', p.returncode, flush=True)
    if check and p.returncode:
        raise RuntimeError(f'{cmd} failed, see {log}')
    return p.returncode

sh('pip install -q "bitsandbytes>=0.45" "peft>=0.17" && python -c "import torch, transformers, peft, bitsandbytes as b; '
   'print(torch.__version__, transformers.__version__, peft.__version__, b.__version__)"', 'install.log', check=True)
print(open(f'{W}/install.log').read()[-300:])

In [ ]:
TRAIN = 'python -m fineprint.training.train'
DDP = 'torchrun --standalone --nproc_per_node 2 -m fineprint.training.train'
if MODE == 'timing':
    sh(f'{DDP} --out {W}/timing-hf2 --max-steps 30 --val-size 32', 'timing_hf2.log')
    sh('pip install -q unsloth', 'install_unsloth.log')
    sh(f'{TRAIN} --backend unsloth --out {W}/timing-unsloth --max-steps 30 --val-size 32', 'timing_unsloth.log', gpus='0')

In [ ]:
if MODE == 'full':
    if BACKEND == 'unsloth':
        sh('pip install -q unsloth', 'install_unsloth.log', check=True)
        cmd = f'{TRAIN} --backend unsloth'
    else:
        cmd = DDP
    sh(f'{cmd} --out {W}/ft ' + ' '.join(EXTRA), 'train.log', gpus='0' if BACKEND == 'unsloth' else '0,1', check=True)
    # Evaluate the adapter on both GPUs at once (same runner and prompt as the B1 baseline).
    # GPU 0: test-500 (compare with every baseline) then the full test set (compare with DeBERTa); GPU 1: LexGLUE.
    ev = f'python -m fineprint.eval.run_hf --system ft_qwen3b_v1 --adapter {W}/ft/adapter --set '
    queues = {'0': ['test-500', 'test'], '1': ['lexglue']}
    jobs = {g: subprocess.Popen(['bash', '-c', '; '.join(f'{ev}{s} >> {W}/eval_gpu{g}.log 2>&1' for s in sets)],
                                env={**ENV, 'CUDA_VISIBLE_DEVICES': g}) for g, sets in queues.items()}
    while any(p.poll() is None for p in jobs.values()):
        time.sleep(120)
        for g in jobs:
            lines = open(f'{W}/eval_gpu{g}.log', errors='replace').read().splitlines()
            print(f'[eval gpu{g}]', lines[-1][:160] if lines else '(starting)', flush=True)
    print({g: p.returncode for g, p in jobs.items()})


In [ ]:
# Package what the Mac needs. Adapter weights go in their own zip (~100 MB); no checkpoints, no base model.
with zipfile.ZipFile(f'{W}/fine-print-results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for pattern in ['results/eval/*.json', 'data/predictions/*/*.jsonl']:
        for f in glob.glob(f'{REPO}/{pattern}'):
            z.write(f, os.path.relpath(f, REPO))
    for f in glob.glob(f'{W}/*/train_summary.json'):
        z.write(f, f'results/training/{os.path.basename(os.path.dirname(f))}.json')
    for f in glob.glob(f'{W}/*.log'):
        z.write(f, 'logs/kaggle_' + os.path.basename(f))
print(zipfile.ZipFile(f'{W}/fine-print-results.zip').namelist())
if os.path.exists(f'{W}/ft/adapter'):
    shutil.make_archive(f'{W}/fine-print-adapter', 'zip', f'{W}/ft/adapter')
for d in glob.glob(f'{W}/*/checkpoint-*') + [REPO]:
    shutil.rmtree(d, ignore_errors=True)  # keep the version's output small and quick to download